In [3]:
import os

In [4]:
%pwd

'c:\\Users\\tukum\\PYthonvscode\\DLproject\\Kidney-Disease-Classification\\research'

In [5]:
os.chdir("../")

In [6]:
%pwd

'c:\\Users\\tukum\\PYthonvscode\\DLproject\\Kidney-Disease-Classification'

In [7]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/entbappy/Kidney-Disease-Classification-MLflow-DVC.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="entbappy"
os.environ["MLFLOW_TRACKING_PASSWORD"]="6824692c47a369aa6f9eac5b10041d5c8edbcef0"

In [8]:
import tensorflow as tf

False


In [9]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [11]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [23]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        return EvaluationConfig(
            path_of_model=Path("artifacts/training/model.h5"),
            training_data=Path("artifacts/data_ingestion/kidney-ct-scan-image"),
            mlflow_uri="https://dagshub.com/entbappy/Kidney-Disease-Classification-MLflow-DVC.mlflow",
            all_params=dict(self.params),
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )

In [24]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [25]:
print("Artifact URI:", mlflow.get_artifact_uri())

Artifact URI: mlflow-artifacts:/9e140c7aaa814695a0bbcbbbbd828f81/8f5af5c5010d4d26a5fc9e4ad61039ef/artifacts


In [26]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        print("🔍 Loading model...")
        self.model = self.load_model(self.config.path_of_model)

        print("📂 Preparing validation data...")
        self._valid_generator()

        print("📊 Evaluating model...")
        self.score = self.model.evaluate(self.valid_generator)

        self.save_score()

    def save_score(self):
        scores = {
            "loss": float(self.score[0]),
            "accuracy": float(self.score[1])
        }
        save_json(path=Path("scores.json"), data=scores)
        print("✅ Scores saved:", scores)

    def log_into_mlflow(self):
        print("🚀 Logging to MLflow...")

        # ✅ FIX 1: correct tracking URI
        mlflow.set_tracking_uri(self.config.mlflow_uri)

        # ✅ FIX 2: end active run (important for notebooks)
        if mlflow.active_run():
            mlflow.end_run()

        tracking_url_type_store = urlparse(
            mlflow.get_tracking_uri()
        ).scheme

        with mlflow.start_run(run_name="evaluation_run"):

            # log params
            mlflow.log_params(self.config.all_params)

            # log metrics
            mlflow.log_metrics({
                "loss": self.score[0],
                "accuracy": self.score[1]
            })

            # log model
            if tracking_url_type_store != "file":
                mlflow.keras.log_model(
                    self.model,
                    "model",
                    registered_model_name="VGG16Model"
                )
            else:
                mlflow.keras.log_model(self.model, "model")

        print("✅ MLflow logging completed!")


In [27]:

    try:
        print("⚙️ Initializing configuration...")
        config = ConfigurationManager()
        eval_config = config.get_evaluation_config()

        evaluation = Evaluation(eval_config)

        evaluation.evaluation()
        evaluation.log_into_mlflow()

        print("🎉 Pipeline completed successfully!")

    except Exception as e:
        print("❌ Error occurred:", e)

⚙️ Initializing configuration...
[2026-05-06 20:44:01,939: INFO: common: yaml file: config\config.yaml loaded successfully]


[2026-05-06 20:44:01,948: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-06 20:44:01,955: INFO: common: created directory at: artifacts]
🔍 Loading model...
📂 Preparing validation data...
Found 139 images belonging to 2 classes.
📊 Evaluating model...
9/9 [==============================] - 27s 3s/step - loss: 4.0237 - accuracy: 0.5036
[2026-05-06 20:44:31,874: INFO: common: json file saved at: scores.json]
✅ Scores saved: {'loss': 4.023709297180176, 'accuracy': 0.5035971403121948}
🚀 Logging to MLflow...


2026/05/06 20:44:34 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-05-06 20:44:36,069: WARNING: save: Found untraced functions such as _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op while saving (showing 5 of 14). These functions will not be directly callable after loading.]
INFO:tensorflow:Assets written to: C:\Users\tukum\AppData\Local\Temp\tmpcc55gwla\model\data\model\assets
[2026-05-06 20:44:37,110: INFO: builder_impl: Assets written to: C:\Users\tukum\AppData\Local\Temp\tmpcc55gwla\model\data\model\assets]


Registered model 'VGG16Model' already exists. Creating a new version of this model...
2026/05/06 20:45:46 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: VGG16Model, version 81
Created version '81' of model 'VGG16Model'.


✅ MLflow logging completed!
🎉 Pipeline completed successfully!
